# 🗺️ Week 3: Getting data for route planning 🗺️

GeoAI workflows are data-driven. Before we can compare possible fiber routes, we need to understand our study area and obtain the spatial data we will work with.

Today we will use a proposed Mombasa–Taveta route in Kenya to work through this process. We will shorten the route for our example, define a study area, and obtain nearby power infrastructure, railways, settlements, and roads from OpenStreetMap (OSM).

We will cover:

* Loading and inspecting spatial data with GeoPandas
* Visualizing a proposed route at different scales
* Converting coordinate reference systems (CRS) for maps and measurements
* Creating a polygon to define where we want data
* Downloading, clipping, and saving OSM data
* Checking whether our results make sense

Run the cells in order, as later cells use variables created earlier. Downloads and background maps need an internet connection. Files are saved in the notebook's working directory; running a save cell again replaces its output layer.

# Learning objectives 🎯

By the end of this notebook, you should be able to:

* Explain the difference between a GeoDataFrame and an individual geometry
* Inspect geometry types and CRS before carrying out spatial processing
* Use a projected CRS when working with distances in metres
* Explain how OSM tags control the data we obtain
* Distinguish mapped road features from a network graph
* Critically assess the coverage and limitations of downloaded data

# 1. Load our packages 📦

Let us first install the packages we need. `%pip` installs into the current notebook kernel. If you have already installed these packages, you can skip this cell.

* `GeoPandas` reads, stores, and processes spatial tables.
* `Shapely` works with individual geometries, such as points, lines, and polygons.
* `OSMnx` downloads OSM features and road networks.
* `Matplotlib` creates our figures, while `Contextily` adds background map tiles.

In [ ]:
# Install the packages into the current notebook kernel
%pip install osmnx geopandas shapely matplotlib contextily

# 2. Read and inspect the proposed route 🔎

A **GeoDataFrame** is a table with a geometry column. Each row represents a spatial feature, while other columns hold its attributes. For example, a route can be stored as a row with a line geometry and a route name.

A **GeoPackage** (`.gpkg`) is a file that can store spatial layers. `gpd.read_file()` reads a layer into a GeoDataFrame, and `to_file()` saves it back to disk.

Let us:

1. Read the proposed route from GitHub.
2. Save a local copy called `route.gpkg`.
3. View its first few rows, geometry types, and CRS.

`head()` shows the first five rows. `.geom_type` tells us whether each geometry is a point, line, or polygon. `.crs` tells us how its coordinates relate to locations on Earth.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

# Load route from GitHub
url = "https://raw.githubusercontent.com/edwardoughton/Agentic-GeoAI/main/data/raw/route1_mombasa_taveta.gpkg"
route = gpd.read_file(url)
route.to_file("route.gpkg")


# Let us inspect the table before making a map
print("CRS:", route.crs)
print("Geometry types:", route.geom_type.value_counts().to_dict())
route.head()

# 3. Put the route on a map 🌍

First, we will show where the route sits within Africa. We read country boundaries from Natural Earth, then select the rows where `CONTINENT` equals `Africa`.

In `world[world["CONTINENT"] == "Africa"]`, the comparison creates a True/False selection for each row. The outer brackets keep the rows marked True. This is a common way to filter both Pandas and GeoPandas tables.

Before adding web map tiles, we convert the layers with `to_crs(epsg=3857)`. This changes their coordinates to Web Mercator. It does not change their real-world locations.

| CRS | How we use it here |
| --- | --- |
| EPSG:4326 | Longitude and latitude in degrees; used for OSMnx polygon queries |
| EPSG:32737 | UTM zone 37S, in metres; used for this route's distance calculations |
| EPSG:3857 | Web Mercator; used to align our layers with background map tiles |

Use `to_crs()` to transform data with a known CRS. Assigning a CRS label with `set_crs()` does not transform coordinates. Web Mercator is useful for our maps, but we use local UTM for measurements.

In [ ]:
# Load Africa boundaries
world = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
)

africa = world[world["CONTINENT"] == "Africa"]

# Reproject to Web Mercator for Contextily
africa_3857 = africa.to_crs(epsg=3857)
route_3857 = route.to_crs(epsg=3857)

# -----------------------------
# MAP 1: Route in Africa context
# -----------------------------
fig, ax = plt.subplots(figsize=(9, 9))

# Africa outline
africa_3857.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=0.7,
    zorder=2
)

# Route
route_3857.plot(
    ax=ax,
    color="red",
    linewidth=4,
    zorder=4
)

# Set Africa-wide extent
minx, miny, maxx, maxy = africa_3857.total_bounds

ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

# Add basemap
ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldStreetMap,
    zorder=1
)

ax.set_axis_off()
ax.set_title("Mombasa–Taveta Route in African Context")

plt.show()

Now let us zoom in to inspect the route itself. We plot the route first, so the map extent follows its location, and then add the background map.

`fig, ax` creates a figure and its plotting area. Passing `ax=ax` puts layers on the same map. `linewidth` controls line thickness, while `zorder` controls which layer appears on top.

In [ ]:
# -----------------------------
# MAP 2: Detailed route map
# -----------------------------
fig, ax = plt.subplots(figsize=(12, 8))

route_3857.plot(
    ax=ax,
    color="blue",
    linewidth=3,
    zorder=2
)

ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldStreetMap,
    zorder=1
)

ax.set_axis_off()
ax.set_title("Mombasa–Taveta Proposed Route")

plt.show()

# 4. Select a shorter route ✂️

We will work with a shorter section to keep our data acquisition example manageable.

1. Project the route to EPSG:32737 so distances are in metres.
2. Extract its line geometry using `route.geometry.iloc[0]`.
3. Create our destination point using **(longitude, latitude)**, and project it to the same CRS.
4. Find the nearest position along the line with `line.project()`.
5. Keep the section from the route's start to that position using `substring()`.
6. Put the resulting geometry back into a GeoDataFrame and save it.

`.geometry` selects the geometry column; `.iloc[0]` selects its first item by position. The extracted Shapely line does not carry a CRS, so we must keep track of its units ourselves. A `GeoSeries` is a single column of geometries with a CRS; we use one to transform the destination point.

This example expects one `LineString`. We check this rather than silently ignoring additional route features. We also use the more easterly endpoint as the start, based on Mombasa's location for this route. This is a case-specific assumption: check the map before applying it elsewhere.

The destination does not have to lie exactly on the route. The cut uses its nearest position **on the line**, rather than drawing a new connection to the destination.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point, LineString
from shapely.ops import substring

# Load the full route
route = gpd.read_file("route.gpkg")

# Project to a metric CRS so distances are in metres
route = route.to_crs(epsg=32737)

# Check that selecting the first geometry will retain the full input route
if len(route) != 1 or route.geom_type.iloc[0] != "LineString":
    raise ValueError("This example needs one LineString; inspect and combine route sections first.")

# Get the route geometry
line = route.geometry.iloc[0]

# Define the destination point
# Point uses (longitude, latitude)
destination = Point(39.471821, -3.865362)

# Convert destination to the same CRS as the route
destination_m = (
    gpd.GeoSeries([destination], crs="EPSG:4326")
    .to_crs(epsg=32737)
    .iloc[0]
)

# Make sure the route runs westward from Mombasa
# Reverse it if necessary
if line.coords[0][0] < line.coords[-1][0]:
    line = LineString(line.coords[::-1])

# Find the distance along the route nearest to the destination
cut_distance = line.project(destination_m)

# Keep only the route from the start to that point
short_route = substring(
    line,
    start_dist=0,
    end_dist=cut_distance
)

if short_route.geom_type != "LineString" or short_route.length == 0:
    raise ValueError("The destination selects no route length. Check the destination and line direction.")
print(f"Selected route length: {short_route.length / 1000:,.2f} km")

# Put it back into a GeoDataFrame
short_route = gpd.GeoDataFrame(
    {"geometry": [short_route]},
    crs="EPSG:32737"
)

# Convert back to latitude/longitude
short_route = short_route.to_crs(epsg=4326)

# Save
short_route.to_file("route1_short.gpkg")

print("Route saved.")

Let us compare the original route (grey) with our selected section (red). This is a simple visual validation step: does the section begin at the expected end, and does it stop near our chosen destination?

In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx

# Reproject both routes for Contextily
route_3857 = route.to_crs(epsg=3857)
short_route_3857 = short_route.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(12, 8))

# Original full route
route_3857.plot(
    ax=ax,
    color="grey",
    linewidth=3,
    alpha=0.7,
    label="Original route",
    zorder=2
)

# New subset route
short_route_3857.plot(
    ax=ax,
    color="red",
    linewidth=4,
    label="Route subset",
    zorder=3
)

# Add basemap
ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldStreetMap,
    zorder=1
)

ax.legend()
ax.set_axis_off()
ax.set_title("Original Mombasa–Taveta Route and Selected Subset")

plt.show()

# Exercise 💬

* What is the difference between `route` and `route.geometry.iloc[0]`?
* Why do we project the destination and route to the same CRS?
* What could go wrong if we always assumed the first coordinate was the start we wanted?

Write down your thoughts so we can discuss them as a class.

# 5. Create our study area 📐

OSMnx needs an area within which to search. Our route is a line, so we will create a **bounding-box polygon** around it.

1. Get the route's minimum and maximum coordinates with `total_bounds`.
2. Expand these limits by 3,000 m on each side.
3. Use `sg.box()` to create a Shapely rectangle.
4. Put the rectangle into a GeoDataFrame with the same projected CRS.
5. Plot it to check the area we will query.

`total_bounds` returns `(minx, miny, maxx, maxy)`. We calculate these limits in metres, so adding `3000` means 3 km.

This rectangle is our search area. It is not a constant-distance buffer following every bend of the route: parts of the rectangle can be much farther from the line. We will use this same rectangle for all downloads and maps below.

In [ ]:
import osmnx as ox
import geopandas as gpd
import shapely.geometry as sg
import matplotlib.pyplot as plt
import contextily as ctx

route = gpd.read_file("route1_short.gpkg")

# Project to UTM (meters)
route = route.to_crs(epsg=32737)

# Get route bounds in meters
minx, miny, maxx, maxy = route.total_bounds

# Expand the route bounds by 3 km on each side
buffer_m = 3000

bbox_geom = sg.box(
    minx - buffer_m,
    miny - buffer_m,
    maxx + buffer_m,
    maxy + buffer_m
)

bbox = gpd.GeoDataFrame(
    {"geometry": [bbox_geom]},
    crs="EPSG:32737"
)

# Convert to Web Mercator for contextily
bbox_3857 = bbox.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(8, 8))

# Transparent bounding box
bbox_3857.plot(
    ax=ax,
    facecolor="none",
    edgecolor="red",
    linewidth=2
)

ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldStreetMap
)

ax.set_axis_off()
ax.set_title("Proposed corridor bounding box")
plt.show()

# 6. Get power infrastructure from OSM ⚡

Now we can pass our study polygon to OSMnx. We first convert the bounding box to EPSG:4326, then extract its geometry with `bbox.geometry.iloc[0]`.

OSM describes features using **tags**, which are keys and values. Our dictionary `{'power': ['line', 'minor_line', 'cable']}` requests features matching any of these power values. The list includes cables as well as overhead lines. For other queries, `True` can request every value of a key. Multiple keys are combined as OR conditions, not AND conditions.

We will:

1. Download matching features with `features_from_polygon()`.
2. Match their CRS to the study area.
3. Use `gpd.clip()` to retain only the geometry inside the rectangle.
4. Save the result as `power_lines.gpkg`.

Clipping trims features at the boundary; selecting features that intersect a polygon would retain their full geometry. The OSMnx result is a GeoDataFrame indexed by OSM element type and ID, which identify the mapped objects.

Downloads may take several minutes. If a request fails or no matching features are returned, inspect the message before continuing. A timeout does not mean the area has no infrastructure, and an empty OSM result does not establish real-world absence. Caching lets successful requests be reused when the query is unchanged.

In [ ]:
ox.settings.use_cache = True # Reuse successful downloads

# Convert bbox to WGS84 for OSMnx
bbox = bbox.to_crs(epsg=4326)

# Download OSM power features
power_lines = ox.features_from_polygon(
    bbox.geometry.iloc[0],
    tags={'power': ['line', 'minor_line', 'cable']}
)

# Make sure CRS matches
power_lines = power_lines.to_crs(bbox.crs)

# Clip all returned geometries to the bounding box
power_lines = gpd.clip(power_lines, bbox)

# Save clipped results
power_lines.to_file("power_lines.gpkg")

Let us add the power infrastructure to our route map. We reuse the polygon from the download so the map shows the area that was actually queried.

Our legend uses `Line2D` for line samples and `Patch` for the polygon outline. These symbols help readers distinguish the proposed route, search boundary, and existing infrastructure.

In [ ]:
import osmnx as ox
import geopandas as gpd
import shapely.geometry as sg
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Load route
route = gpd.read_file("route1_short.gpkg")

# Project to UTM for distance/buffer work
route = route.to_crs(epsg=32737)

# Reuse bbox from the download; keep the search area consistent

# Reproject everything to Web Mercator for contextily
route_3857 = route.to_crs(epsg=3857)
bbox_3857 = bbox.to_crs(epsg=3857)
power_lines_3857 = power_lines.to_crs(epsg=3857)

# Create map
fig, ax = plt.subplots(figsize=(10, 10))

# Plot bounding box
bbox_3857.plot(
    ax=ax,
    facecolor="none",
    edgecolor="red",
    linewidth=2,
    zorder=3
)

# Plot power lines
power_lines_3857.plot(
    ax=ax,
    color="orange",
    linewidth=1.5,
    zorder=4
)

# Plot proposed corridor route
route_3857.plot(
    ax=ax,
    color="blue",
    linewidth=3,
    zorder=5
)

# Add basemap
ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldStreetMap,
    zorder=1
)

# Custom legend
legend_elements = [
    Line2D(
        [0], [0],
        color="blue",
        linewidth=3,
        label="Proposed corridor route"
    ),
    Patch(
        facecolor="none",
        edgecolor="red",
        linewidth=2,
        label="Corridor polygon"
    ),
    Line2D(
        [0], [0],
        color="orange",
        linewidth=1.5,
        label="Power lines"
    )
]

ax.legend(
    handles=legend_elements,
    loc="upper right"
)

ax.set_axis_off()
ax.set_title("Proposed Fiber Corridor and Power Infrastructure")

plt.show()

# 7. Repeat for railways 🚆

The workflow is the same, but our tags now request `rail`, `narrow_gauge`, and `light_rail` values under the `railway` key. Notice how changing one dictionary lets us obtain a different type of infrastructure for the same study area.

We clip the result and save it as `rail_lines.gpkg` before adding it to our map.

In [ ]:
# Convert bbox to WGS84 for OSMnx
bbox = bbox.to_crs(epsg=4326)

# Download OSM railway features
rail_lines = ox.features_from_polygon(
    bbox.geometry.iloc[0],
    tags={'railway': ['rail', 'narrow_gauge', 'light_rail']}
)

# Make sure CRS matches
rail_lines = rail_lines.to_crs(bbox.crs)

# Clip all returned geometries to the bounding box
rail_lines = gpd.clip(rail_lines, bbox)

# Save clipped results
rail_lines.to_file("rail_lines.gpkg")

Now we can view the route, power infrastructure, and railways together.

The geometry filter uses `.isin(["LineString", "MultiLineString"])` to retain line features for plotting. A `LineString` is one line; a `MultiLineString` contains multiple line parts. OSM feature queries can contain different geometry types, so checking them is useful before deciding how to display or analyze the data. The filter here affects the plotting copies, not the saved source layers.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Reproject everything to Web Mercator for contextily
route_3857 = route.to_crs(epsg=3857)
bbox_3857 = bbox.to_crs(epsg=3857)
power_lines_3857 = power_lines.to_crs(epsg=3857)
rail_lines_3857 = rail_lines.to_crs(epsg=3857)

# Optional: keep only line geometries
power_lines_3857 = power_lines_3857[
    power_lines_3857.geometry.geom_type.isin(["LineString", "MultiLineString"])
]

rail_lines_3857 = rail_lines_3857[
    rail_lines_3857.geometry.geom_type.isin(["LineString", "MultiLineString"])
]

# Create map
fig, ax = plt.subplots(figsize=(10, 10))

# Corridor polygon
bbox_3857.plot(
    ax=ax,
    facecolor="none",
    edgecolor="red",
    linewidth=2,
    zorder=3
)

# Power lines
power_lines_3857.plot(
    ax=ax,
    color="orange",
    linewidth=1.5,
    zorder=4
)

# Rail lines
rail_lines_3857.plot(
    ax=ax,
    color="black",
    linewidth=1.8,
    zorder=5
)

# Proposed corridor route
route_3857.plot(
    ax=ax,
    color="blue",
    linewidth=3,
    zorder=6
)

# Basemap
ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldStreetMap,
    zorder=1
)

# Legend
legend_elements = [
    Line2D(
        [0], [0],
        color="blue",
        linewidth=3,
        label="Proposed corridor route"
    ),
    Patch(
        facecolor="none",
        edgecolor="red",
        linewidth=2,
        label="Corridor polygon"
    ),
    Line2D(
        [0], [0],
        color="orange",
        linewidth=1.5,
        label="Power lines"
    ),
    Line2D(
        [0], [0],
        color="black",
        linewidth=1.8,
        label="Rail lines"
    )
]

ax.legend(
    handles=legend_elements,
    loc="upper right"
)

ax.set_axis_off()
ax.set_title("Proposed Fiber Corridor with Power and Rail Infrastructure")

plt.show()

# 8. Add settlements 🏘️

Next, let us request cities, towns, villages, hamlets, and suburbs using the `place` key. These mapped places can help us explore where a proposed fiber route might serve communities.

We follow the same download, CRS, clip, and save steps. Place features are not population estimates, and their coverage or classification may vary between locations.

In [ ]:
# Convert bbox to WGS84 for OSMnx
bbox = bbox.to_crs(epsg=4326)

# Download OSM settlement features
settlements = ox.features_from_polygon(
    bbox.geometry.iloc[0],
    tags={'place': ['city', 'town', 'village', 'hamlet', 'suburb']}
)

# Make sure CRS matches
settlements = settlements.to_crs(bbox.crs)

# Clip all returned geometries to the bounding box
settlements = gpd.clip(settlements, bbox)

# Save clipped results
settlements.to_file("settlements.gpkg")

Some settlements are mapped as points, while others may be polygons. To draw them with a consistent point symbol, we create a plotting copy using `.copy()` and calculate representative points. For a polygon, a representative point lies inside it; it is a display location, not necessarily the town centre.

Keeping a separate plotting copy means the saved settlement geometries retain their original form. We then add these locations in purple to the same map.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Reproject all layers to Web Mercator
route_3857 = route.to_crs(epsg=3857)
bbox_3857 = bbox.to_crs(epsg=3857)
power_lines_3857 = power_lines.to_crs(epsg=3857)
rail_lines_3857 = rail_lines.to_crs(epsg=3857)
settlements_3857 = settlements.to_crs(epsg=3857)

# Keep only line geometries for infrastructure
power_lines_3857 = power_lines_3857[
    power_lines_3857.geometry.geom_type.isin(["LineString", "MultiLineString"])
]

rail_lines_3857 = rail_lines_3857[
    rail_lines_3857.geometry.geom_type.isin(["LineString", "MultiLineString"])
]

# Convert settlement geometries to points for plotting
settlements_plot = settlements_3857.copy()

settlements_plot["geometry"] = settlements_plot.geometry.representative_point()

# Create map
fig, ax = plt.subplots(figsize=(10, 10))

# Corridor polygon
bbox_3857.plot(
    ax=ax,
    facecolor="none",
    edgecolor="red",
    linewidth=2,
    zorder=3
)

# Power lines
power_lines_3857.plot(
    ax=ax,
    color="orange",
    linewidth=1.5,
    zorder=4
)

# Rail lines
rail_lines_3857.plot(
    ax=ax,
    color="black",
    linewidth=1.8,
    zorder=5
)

# Settlements
settlements_plot.plot(
    ax=ax,
    color="purple",
    markersize=35,
    edgecolor="white",
    linewidth=0.5,
    zorder=6
)

# Proposed corridor route
route_3857.plot(
    ax=ax,
    color="blue",
    linewidth=3,
    zorder=7
)

# Basemap
ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldStreetMap,
    zorder=1
)

# Legend
legend_elements = [
    Line2D(
        [0], [0],
        color="blue",
        linewidth=3,
        label="Proposed corridor route"
    ),
    Patch(
        facecolor="none",
        edgecolor="red",
        linewidth=2,
        label="Corridor polygon"
    ),
    Line2D(
        [0], [0],
        color="orange",
        linewidth=1.5,
        label="Power lines"
    ),
    Line2D(
        [0], [0],
        color="black",
        linewidth=1.8,
        label="Rail lines"
    ),
    Line2D(
        [0], [0],
        marker="o",
        linestyle="None",
        markerfacecolor="purple",
        markeredgecolor="white",
        markersize=8,
        label="Settlements"
    )
]

ax.legend(
    handles=legend_elements,
    loc="upper right"
)

ax.set_axis_off()
ax.set_title(
    "Proposed Fiber Corridor with Power, Rail and Settlements"
)

plt.show()

# 9. Obtain the road network 🛣️

Roads give us an opportunity to use a different OSMnx function. `graph_from_polygon()` returns a **network graph**: nodes represent network junctions or endpoints, and edges represent connections between them. `network_type="drive"` requests a network for driving, rather than every path or road-tagged feature.

1. Download the graph for our polygon.
2. Convert its nodes and edges to GeoDataFrames with `graph_to_gdfs()`.
3. Clip the road geometries to our study area.
4. Save the road layer as `roads.gpkg`.

`nodes, roads = ...` assigns the two returned tables to separate variables. The graph `G` retains connectivity, while `roads` is the spatial table of edges used for mapping. Clipping that table does not update the graph. Use `G` for network analysis; the exported road layer alone is not a complete saved graph.

Directed edges can represent both travel directions along the same road, so row counts are not counts of unique physical roads. Some edge attributes also contain lists, which we convert to text before saving to GeoPackage.

In [ ]:
# Convert bbox to WGS84 for OSMnx
bbox = bbox.to_crs(epsg=4326)

# Download road network
G = ox.graph_from_polygon(
    bbox.geometry.iloc[0],
    network_type="drive"
)

# Convert network to GeoDataFrames
nodes, roads = ox.graph_to_gdfs(G)

# Clip roads to the exact bounding box
roads = gpd.clip(roads, bbox)

# Save mapped road edges, keeping the graph identifiers as columns
roads_export = roads.reset_index()
for column in roads_export.columns:
    if column != "geometry":
        roads_export[column] = roads_export[column].map(
            lambda value: str(value) if isinstance(value, (list, dict, tuple)) else value
        )
roads_export.to_file("roads.gpkg", index=False)

print(roads.head())

# 10. Bring our layers together 🗺️

Finally, we will map the roads, power infrastructure, railways, settlements, and proposed fiber route together. Each layer is converted to EPSG:3857 to align with the background map.

Notice the drawing order: roads are drawn first, with the proposed route on top so it remains visible. This map helps us inspect spatial relationships; proximity alone does not establish that infrastructure can be shared or that a route is feasible.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Reproject everything to Web Mercator for contextily
route_3857 = route.to_crs(epsg=3857)
bbox_3857 = bbox.to_crs(epsg=3857)
power_lines_3857 = power_lines.to_crs(epsg=3857)
rail_lines_3857 = rail_lines.to_crs(epsg=3857)
settlements_3857 = settlements.to_crs(epsg=3857)
roads_3857 = roads.to_crs(epsg=3857)

# Keep only line geometries
power_lines_3857 = power_lines_3857[
    power_lines_3857.geometry.geom_type.isin(["LineString", "MultiLineString"])
]

rail_lines_3857 = rail_lines_3857[
    rail_lines_3857.geometry.geom_type.isin(["LineString", "MultiLineString"])
]

roads_3857 = roads_3857[
    roads_3857.geometry.geom_type.isin(["LineString", "MultiLineString"])
]

# Convert settlements to points for plotting
settlements_plot = settlements_3857.copy()

settlements_plot["geometry"] = settlements_plot.geometry.representative_point()

# Create map
fig, ax = plt.subplots(figsize=(12, 10))

# -------------------------
# Corridor polygon
# -------------------------
bbox_3857.plot(
    ax=ax,
    facecolor="none",
    edgecolor="red",
    linewidth=2,
    zorder=3
)

# -------------------------
# Road network
# -------------------------
roads_3857.plot(
    ax=ax,
    color="grey",
    linewidth=0.6,
    alpha=0.8,
    zorder=4
)

# -------------------------
# Power lines
# -------------------------
power_lines_3857.plot(
    ax=ax,
    color="orange",
    linewidth=1.5,
    zorder=5
)

# -------------------------
# Rail lines
# -------------------------
rail_lines_3857.plot(
    ax=ax,
    color="black",
    linewidth=1.8,
    zorder=6
)

# -------------------------
# Settlements
# -------------------------
settlements_plot.plot(
    ax=ax,
    color="purple",
    markersize=30,
    edgecolor="white",
    linewidth=0.5,
    zorder=7
)

# -------------------------
# Proposed fiber route
# -------------------------
route_3857.plot(
    ax=ax,
    color="blue",
    linewidth=3,
    zorder=8
)

# -------------------------
# Basemap
# -------------------------
ctx.add_basemap(
    ax,
    source=ctx.providers.Esri.WorldStreetMap,
    zorder=1
)

# -------------------------
# Legend
# -------------------------
legend_elements = [

    Line2D(
        [0], [0],
        color="blue",
        linewidth=3,
        label="Proposed corridor route"
    ),

    Patch(
        facecolor="none",
        edgecolor="red",
        linewidth=2,
        label="Corridor polygon"
    ),

    Line2D(
        [0], [0],
        color="grey",
        linewidth=1,
        label="Road network"
    ),

    Line2D(
        [0], [0],
        color="orange",
        linewidth=1.5,
        label="Power lines"
    ),

    Line2D(
        [0], [0],
        color="black",
        linewidth=1.8,
        label="Rail lines"
    ),

    Line2D(
        [0], [0],
        marker="o",
        linestyle="None",
        markerfacecolor="purple",
        markeredgecolor="white",
        markersize=8,
        label="Settlements"
    )
]

ax.legend(
    handles=legend_elements,
    loc="upper right"
)

ax.set_axis_off()

ax.set_title(
    "Proposed Fiber Corridor and Existing Infrastructure"
)

plt.show()

# 11. Check what we obtained ✅

Before moving on, let us inspect the number of rows, CRS, and geometry types in each downloaded layer. A dictionary lets us associate a readable name with each GeoDataFrame, then repeat the same checks in a loop.

These counts describe mapped features or network edges. They do not tell us how many complete power circuits, railway routes, or unique roads exist. View a few attributes with `.head()` and compare their locations with the maps.

In [ ]:
# Inspect each downloaded layer
layers = {"Power": power_lines, "Rail": rail_lines, "Settlements": settlements, "Roads": roads}
for name, data in layers.items():
    print(f"{name}: {len(data):,} rows | CRS: {data.crs}")
    print("Geometry types:", data.geom_type.value_counts().to_dict())
    print("Missing geometries:", int(data.geometry.isna().sum()))
    print("Empty geometries:", int(data.geometry.is_empty.sum()))
    print("Invalid geometries:", int((data.geometry.notna() & ~data.geometry.is_valid).sum()))

# Exercise 💬

* Explain the difference between a GeoPackage, a GeoDataFrame, and a Shapely geometry in your own words.
* What would change if we used a buffer following the route instead of a bounding box?
* Change the 3 km margin, then rerun the downloads and maps. How does the study area affect what we obtain?
* Try requesting power substations. What tag and geometry types would you need to consider?
* Identify one place where the map suggests infrastructure is close to the proposed route. What further information would you need before recommending its use?
* How would you investigate an empty result or a failed download?

Using a generative AI tool of your choice, describe one of these changes in natural language and try the suggested code. Document any errors, explain how you checked the result, and bring your observations to our class discussion.

# Data sources and further reading 📚

* [GeoPandas introduction](https://geopandas.org/en/stable/getting_started/introduction.html): spatial tables, geometry columns, and common operations.
* [OSMnx documentation](https://osmnx.readthedocs.io/en/stable/user-reference.html): feature queries, tags, and road networks.
* [Natural Earth](https://www.naturalearthdata.com/): country boundaries for our overview map.
* [OpenStreetMap contributors](https://www.openstreetmap.org/copyright): infrastructure and settlement data. Retain attribution when sharing maps or data.

OSM coverage varies by location and feature type. Treat our downloads as mapped evidence to inspect, rather than a complete inventory. Background map tiles are a separate service from the OSM feature downloads, so one can fail while the other works.